# Fairness Analysis of Resume Matching Scores

In this notebook, we analyze the SBERT matching scores generated in the previous step. The goal is to check whether the matching score changes when only one demographic signal in the resume is modified.

Each resume has an original version and counterfactual versions where only the name, pronouns, or university are changed. By comparing the original score with the changed version score, I can measure whether the model is sensitive to these demographic signals.

This step helps audit bias in the resume-job matching pipeline.

In [1]:
# Import libraries for fairness analysis
import os
import pandas as pd

# Create a results folder to save the output files
os.makedirs("results", exist_ok=True)

In [6]:
# Upload the SBERT scores file generated from the previous notebook
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

Saving sbert_scores (1).csv to sbert_scores (1) (1).csv
Uploaded files: ['sbert_scores (1) (1).csv']


In [7]:
# Load the SBERT matching scores
scores = pd.read_csv("sbert_scores (1).csv")
print("Scores shape:", scores.shape)
display(scores.head())
display(scores["version"].value_counts())
display(scores["changed_signal"].value_counts())

Scores shape: (200, 6)


,resume_id,version,changed_signal,job_id,job_title,similarity_score
0,R1,original,none,J1,Software Engineer,0.529075
1,R1,original,none,J2,Financial Analyst,0.170317
2,R1,original,none,J3,Marketing Coordinator,0.319204
3,R1,original,none,J4,Clinical Data Analyst,0.326272
4,R1,original,none,J5,Academic Advisor,0.337290


,count
version,
original,50
name_changed,50
pronoun_changed,50
university_changed,50


,count
changed_signal,
none,50
name,50
pronoun,50
university,50


In [8]:
# Separate the original resume scores from the counterfactual changed resume scores
original_scores = scores[scores["version"] == "original"].copy()
changed_scores = scores[scores["version"] != "original"].copy()
# Keep only the columns needed from the original scores
original_scores = original_scores[[
    "resume_id",
    "job_id",
    "job_title",
    "similarity_score"
]]
# Rename the original score column for easier comparison
original_scores = original_scores.rename(columns={
    "similarity_score": "original_score"
})
# Rename changed score column
changed_scores = changed_scores.rename(columns={
    "similarity_score": "changed_score"
})
print("Original scores:", original_scores.shape)
print("Changed scores:", changed_scores.shape)
display(original_scores.head())
display(changed_scores.head())

Original scores: (50, 4)
Changed scores: (150, 6)


,resume_id,job_id,job_title,original_score
0,R1,J1,Software Engineer,0.529075
1,R1,J2,Financial Analyst,0.170317
2,R1,J3,Marketing Coordinator,0.319204
3,R1,J4,Clinical Data Analyst,0.326272
4,R1,J5,Academic Advisor,0.337290


,resume_id,version,changed_signal,job_id,job_title,changed_score
5,R1,name_changed,name,J1,Software Engineer,0.504113
6,R1,name_changed,name,J2,Financial Analyst,0.191474
7,R1,name_changed,name,J3,Marketing Coordinator,0.329989
8,R1,name_changed,name,J4,Clinical Data Analyst,0.366445
9,R1,name_changed,name,J5,Academic Advisor,0.336263


In [9]:
# Match each changed resume score with its corresponding original resume score
# The comparison is based on the same resume_id and job_id
comparison = changed_scores.merge(
    original_scores,
    on=["resume_id", "job_id", "job_title"],
    how="left"
)
# Calculate how much the score changed after modifying one demographic signal
comparison["score_difference"] = comparison["changed_score"] - comparison["original_score"]
# Absolute difference helps measure the size of the change without considering direction
comparison["absolute_difference"] = comparison["score_difference"].abs()
display(comparison.head())
print("Comparison rows:", comparison.shape)

,resume_id,version,changed_signal,job_id,job_title,changed_score,original_score,score_difference,absolute_difference
0,R1,name_changed,name,J1,Software Engineer,0.504113,0.529075,-0.024961,0.024961
1,R1,name_changed,name,J2,Financial Analyst,0.191474,0.170317,0.021157,0.021157
2,R1,name_changed,name,J3,Marketing Coordinator,0.329989,0.319204,0.010785,0.010785
3,R1,name_changed,name,J4,Clinical Data Analyst,0.366445,0.326272,0.040173,0.040173
4,R1,name_changed,name,J5,Academic Advisor,0.336263,0.337290,-0.001027,0.001027


Comparison rows: (150, 9)


In [10]:
# Summarize how much the score changed for each demographic signal
fairness_summary = comparison.groupby("changed_signal").agg(
    average_score_difference=("score_difference", "mean"),
    average_absolute_difference=("absolute_difference", "mean"),
    max_absolute_difference=("absolute_difference", "max"),
    min_score_difference=("score_difference", "min"),
    max_score_difference=("score_difference", "max")
).reset_index()
display(fairness_summary)

,changed_signal,average_score_difference,average_absolute_difference,max_absolute_difference,min_score_difference,max_score_difference
0,name,0.007107,0.021594,0.087211,-0.048753,0.087211
1,pronoun,0.005440,0.006963,0.016791,-0.008120,0.016791
2,university,-0.000921,0.012347,0.085761,-0.085761,0.039752


## Interpretation of Fairness Results

The fairness summary shows that changing demographic signals can slightly change the resume-job matching scores, even though the main qualifications and experience remain the same.

Among the tested signals, name changes produced the largest average absolute difference in similarity score. University changes also caused some score variation, while pronoun changes had the smallest average effect.

These results suggest that the SBERT-based matching pipeline is mostly stable, but it is not completely insensitive to demographic-related text changes. Since the dataset is small, these findings should be treated as an audit-style observation rather than a final conclusion about real-world hiring bias.

In [11]:
# Save the detailed comparison and summary results
comparison.to_csv("results/fairness_comparison.csv", index=False)
fairness_summary.to_csv("results/fairness_summary.csv", index=False)
print("Fairness comparison saved to results/fairness_comparison.csv")
print("Fairness summary saved to results/fairness_summary.csv")

Fairness comparison saved to results/fairness_comparison.csv
Fairness summary saved to results/fairness_summary.csv


In [12]:
# Download the fairness result files so they can be uploaded to GitHub
from google.colab import files
files.download("results/fairness_comparison.csv")
files.download("results/fairness_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>